## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [15]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

%matplotlib inline

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
effect_size = sms.proportion_effectsize(0.20, 0.19)

required_n = sms.NormalIndPower().solve_power(
    effect_size,
    power=0.8,
    alpha=0.05,
    ratio=1
)

required_n = ceil(required_n)

print("Користувачів у кожній групі:", required_n)
print("Користувачів сумарно:", required_n * 2)

Користувачів у кожній групі: 24638
Користувачів сумарно: 49276


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [18]:
df = pd.read_csv('/content/drive/MyDrive/Data/cookie_cats.csv')
df.head()


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [20]:
retention_by_version = df.groupby("version")["retention_7"].mean()
print(retention_by_version)

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


Ми хочемо перевірити, чи впливає переміщення воріт з 30-го рівня на 40-й на показник утримання користувачів через 7 днів після встановлення гри (retention_7).

Припускаємо, що версія gate_40 може забезпечувати краще утримання користувачів, ніж gate_30, тому формулюємо гіпотези таким чином:

H0: p30 = p40
	​

Ha: p30 < p40

Ми також встановимо рівень значущості 95%

𝛼=0.05
	​


3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [28]:
control = df[df["version"] == "gate_30"]["retention_7"]
treatment = df[df["version"] == "gate_40"]["retention_7"]

count = [
    control.sum(),
    treatment.sum()
]

nobs = [
    len(control),
    len(treatment)
]

# z-test
stat, pval = proportions_ztest(count, nobs, alternative="smaller")

# Довірчі інтервали
confint_control = proportion_confint(count[0], nobs[0], alpha=0.05, method="normal")
confint_treatment = proportion_confint(count[1], nobs[1], alpha=0.05, method="normal")

print(f"Z statistic = {stat:.3f}")
print(f"p-value = {pval:.4f}")
print(f"Довірчий інтервал 95% для групи control: [{confint_control[0]:.4f}, {confint_control[1]:.4f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{confint_treatment[0]:.4f}, {confint_treatment[1]:.4f}]")

Z statistic = 3.164
p-value = 0.9992
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]


**Висновки**

Ми перевіряли гіпотезу про те, що версія gate_40 забезпечує краще утримання користувачів через 7 днів після встановлення гри.

Оскільки p-value = 0.9992, що значно перевищує рівень значущості 0.05, ми не маємо підстав відхиляти нульову гіпотезу. Отже, отримані дані не підтверджують припущення про те, що версія gate_40 покращує показник retention_7.

Довірчі інтервали показують, що утримання користувачів є вищим у версії gate_30.

Довірчі інтервали не перетинаються, що свідчить про наявність помітної різниці між версіями гри. Оскільки інтервал для gate_30 повністю розташований вище за інтервал gate_40, це вказує на те, що версія з воротами на 30 рівні демонструє краще утримання користувачів через 7 днів.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


Гіпотези для тесту Хі-квадрат:

H0: версія гри та утримання користувача на 7 день є незалежними.

Ha: між версією гри та утриманням користувача на 7 день існує залежність.

In [31]:
# Формуємо таблицю спряженості

table = pd.crosstab(df["version"], df["retention_7"])
print(table)

retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279


In [32]:
# Виконуємо χ²-тест

chi2, p, dof, expected = stats.chi2_contingency(table)

print(f"χ² = {chi2:.3f}")
print(f"p-value = {p:.5f}")
print(f"Ступені свободи = {dof}")

print("Очікувані частоти:\n", expected)

χ² = 9.959
p-value = 0.00160
Ступені свободи = 1
Очікувані частоти:
 [[36382.90257127  8317.09742873]
 [37025.09742873  8463.90257127]]


**Висновки**

Оскільки p-value = 0.00160, що є меншим за рівень значущості 0.05, ми відхиляємо нульову гіпотезу.

Отже, між версією гри та утриманням користувачів через 7 днів після встановлення існує статистично значуща залежність. Це означає, що зміна розташування воріт впливає на поведінку користувачів і на показник retention_7.